# ERP 입력 대상 분류 및 업로드 데이터 생성

정상 분개장과 오류 분개장을 결합해 운영 상황을 구성한다.
검증 결과에 따라 전표를 입력 가능 또는 검토 필요 상태로 분류한다.

## 1. 정상 분개장과 오류 분개장 결합

정상 거래 10건과 오류 거래 10건을 결합해 총 20건의 운영 데이터를 만든다.
두 데이터에서 전표번호와 증빙번호가 겹치지 않도록 정상 거래의 식별자를 변경한다.

In [1]:
# 데이터 처리와 경로 설정에 필요한 라이브러리
import pandas as pd
from pathlib import Path


# 현재 실행 위치를 기준으로 프로젝트 루트 설정
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 정상 분개장과 오류 분개장 경로
normal_journal_path = (
    project_root
    / "data"
    / "raw"
    / "journal_sample.xlsx"
)

error_journal_path = (
    project_root
    / "data"
    / "raw"
    / "journal_with_errors.xlsx"
)

# 04번 노트북에서 저장한 오류 검증 결과 경로
validation_result_path = (
    project_root
    / "data"
    / "processed"
    / "validation_result.csv"
)

# 세 파일 불러오기
normal_journal = pd.read_excel(
    normal_journal_path,
    sheet_name="분개장"
)

error_journal = pd.read_excel(
    error_journal_path,
    sheet_name="분개장"
)

validation_result = pd.read_csv(
    validation_result_path
)

# 정상 데이터의 전표번호를 101번부터 새롭게 부여
normal_journal["voucher_id"] = [
    f"JV202608{number:03d}"
    for number in range(101, 111)
]

# 오류 데이터와 증빙번호가 겹치지 않도록 정상 데이터에 접두어 추가
normal_journal["evidence_no"] = (
    "NORMAL-"
    + normal_journal["evidence_no"].astype(str)
)

# 정상 거래와 오류 거래를 하나의 운영 데이터로 결합
operation_journal = pd.concat(
    [
        normal_journal,
        error_journal
    ],
    ignore_index=True
)

print("정상 거래 수:", len(normal_journal))
print("오류 거래 수:", len(error_journal))
print("전체 운영 거래 수:", len(operation_journal))
print(
    "중복 전표번호 수:",
    operation_journal["voucher_id"].duplicated().sum()
)

operation_journal[
    [
        "voucher_id",
        "transaction_date",
        "partner_code",
        "evidence_no",
        "description",
        "total_amount"
    ]
]

정상 거래 수: 10
오류 거래 수: 10
전체 운영 거래 수: 20
중복 전표번호 수: 0


,voucher_id,transaction_date,partner_code,evidence_no,description,total_amount
0,JV202608101,2026-08-01,V001,NORMAL-TAX-202608-001,고무 원재료 외상 매입,5500000
1,JV202608102,2026-08-02,V003,NORMAL-CARD-202608-001,생산 작업용 장갑 구매,220000
2,JV202608103,2026-08-03,V004,NORMAL-TAX-202608-002,제품 포장재 외상 구매,880000
3,JV202608104,2026-08-05,V005,NORMAL-TAX-202608-003,제품 납품 운송비,660000
4,JV202608105,2026-08-08,V006,NORMAL-TAX-202608-004,공장 전기요금,1650000
5,JV202608106,2026-08-10,V007,NORMAL-TAX-202608-005,생산설비 외상 구매,22000000
6,JV202608107,2026-08-12,V008,NORMAL-TAX-202608-006,회계 자문 수수료,1100000
7,JV202608108,2026-08-15,C001,NORMAL-SALE-202608-001,자동차용 고무부품 외상 판매,13200000
8,JV202608109,2026-08-18,C002,NORMAL-SALE-202608-002,자동차용 씰링부품 외상 판매,8800000
9,JV202608110,2026-08-20,V001,NORMAL-BANK-202608-001,원재료 외상매입금 지급,5500000


## 2. 검증 결과에 따른 처리 상태 분류

전표별 오류 개수와 오류 유형을 집계한다.
오류가 없는 전표는 입력 가능, 하나 이상의 오류가 있는 전표는 검토 필요로 분류한다.

In [2]:
# 전표별로 발견된 오류 개수와 오류 유형을 집계
error_summary = (
    validation_result
    .groupby("voucher_id")
    .agg(
        error_count=("error_type", "count"),
        error_types=(
            "error_type",
            lambda values: ", ".join(sorted(set(values)))
        )
    )
    .reset_index()
)

# 운영 데이터에 전표별 오류 정보를 연결
classified_journal = operation_journal.merge(
    error_summary,
    on="voucher_id",
    how="left"
)

# 오류 결과가 없는 정상 전표의 오류 개수를 0으로 처리
classified_journal["error_count"] = (
    classified_journal["error_count"]
    .fillna(0)
    .astype(int)
)

# 오류 결과가 없는 정상 전표의 오류 유형을 별도 문구로 표시
classified_journal["error_types"] = (
    classified_journal["error_types"]
    .fillna("없음")
)

# 오류 개수에 따라 처리 상태 결정
classified_journal["processing_status"] = (
    classified_journal["error_count"]
    .apply(
        lambda count: "입력 가능"
        if count == 0
        else "검토 필요"
    )
)

# 처리 상태별 거래 수 집계
status_summary = (
    classified_journal["processing_status"]
    .value_counts()
    .rename_axis("processing_status")
    .reset_index(name="transaction_count")
)

print("처리 상태별 거래 수")
display(status_summary)

# 전표별 분류 결과 확인
classified_journal[
    [
        "voucher_id",
        "description",
        "total_amount",
        "error_count",
        "error_types",
        "processing_status"
    ]
]

처리 상태별 거래 수


,processing_status,transaction_count
0,입력 가능,10
1,검토 필요,10


,voucher_id,description,total_amount,error_count,error_types,processing_status
0,JV202608101,고무 원재료 외상 매입,5500000,0,없음,입력 가능
1,JV202608102,생산 작업용 장갑 구매,220000,0,없음,입력 가능
2,JV202608103,제품 포장재 외상 구매,880000,0,없음,입력 가능
3,JV202608104,제품 납품 운송비,660000,0,없음,입력 가능
4,JV202608105,공장 전기요금,1650000,0,없음,입력 가능
5,JV202608106,생산설비 외상 구매,22000000,0,없음,입력 가능
6,JV202608107,회계 자문 수수료,1100000,0,없음,입력 가능
7,JV202608108,자동차용 고무부품 외상 판매,13200000,0,없음,입력 가능
8,JV202608109,자동차용 씰링부품 외상 판매,8800000,0,없음,입력 가능
9,JV202608110,원재료 외상매입금 지급,5500000,0,없음,입력 가능


## 3. ERP 입력 대상과 검토 필요 전표 분리

처리 상태가 입력 가능한 전표와 담당자의 확인이 필요한 전표를 분리한다.
검토 필요 전표에는 오류가 발생한 열과 상세 사유를 연결한다.

In [3]:
# 오류가 없어 ERP 입력이 가능한 전표만 추출
ready_journal = (
    classified_journal.loc[
        classified_journal["processing_status"] == "입력 가능"
    ]
    .copy()
    .reset_index(drop=True)
)

# 하나 이상의 오류가 있어 담당자 확인이 필요한 전표 추출
review_journal = (
    classified_journal.loc[
        classified_journal["processing_status"] == "검토 필요"
    ]
    .copy()
    .reset_index(drop=True)
)

# 상세 오류 유형의 열 이름을 명확하게 변경
validation_details = validation_result[
    [
        "voucher_id",
        "column",
        "error_type",
        "detail"
    ]
].rename(
    columns={
        "error_type": "error_type_detail"
    }
)

# 검토 필요 전표에 오류가 발생한 열과 상세 설명 연결
review_queue = review_journal.merge(
    validation_details,
    on="voucher_id",
    how="left"
)

print("ERP 입력 가능 전표 수:", len(ready_journal))
print("검토 필요 전표 수:", len(review_journal))
print("검토 대기열 오류 수:", len(review_queue))

# 담당자가 확인할 핵심 정보 출력
review_queue[
    [
        "voucher_id",
        "transaction_date",
        "department_code",
        "partner_code",
        "description",
        "column",
        "error_type_detail",
        "detail"
    ]
]

ERP 입력 가능 전표 수: 10
검토 필요 전표 수: 10
검토 대기열 오류 수: 10


,voucher_id,transaction_date,department_code,partner_code,description,column,error_type_detail,detail
0,JV202608001,2026-08-01,D002,V001,고무 원재료 외상 매입,debit_account_1,존재하지 않는 계정과목,9999 계정과목 기준표에 없음
1,JV202608002,2026-08-02,D003,V999,생산 작업용 장갑 구매,partner_code,존재하지 않는 거래처,V999 거래처 기준표에 없음
2,JV202608003,2026-08-03,D999,V004,제품 포장재 외상 구매,department_code,존재하지 않는 부서,D999 부서 기준표에 없음
3,JV202608004,2026-08-05,D007,V005,제품 납품 운송비,debit_amount_1,차변·대변 불일치,"차변 670,000원 / 대변 660,000원"
4,JV202608005,2026-08-08,D003,V006,공장 전기요금,vat_amount,부가세 계산 오류,"입력 100,000원 / 예상 150,000원"
5,JV202608006,2026-08-10,D002,V007,생산설비 외상 구매,evidence_no,필수값 누락,evidence_no 값 누락
6,JV202608007,2026-08-12,D001,V008,회계 자문 수수료,evidence_no,증빙번호 중복,TAX-202608-004: 중복 증빙
7,JV202608008,2026-09-05,D006,C001,자동차용 고무부품 외상 판매,transaction_date,회계기간 이탈,2026-09-05: 8월 회계기간 아님
8,JV202608009,2026-08-18,D006,C002,NaN,description,필수값 누락,description 값 누락
9,JV202608010,2026-08-20,D001,V001,원재료 외상매입금 지급,total_amount,전표 합계 불일치,"입력 5,400,000원 / 분개 5,500,000원"


## 4. 입력 가능한 전표를 ERP 업로드 형식으로 변환

입력 가능 상태의 가로형 분개를 계정과목별 세로형 데이터로 변환한다.
차변과 대변의 각 계정과목을 개별 행으로 분리하고 계정과목명을 연결한다.

In [4]:
# 계정과목명을 연결하기 위해 계정과목 기준표 불러오기
accounts_path = (
    project_root
    / "data"
    / "master"
    / "accounts.csv"
)

accounts = pd.read_csv(accounts_path)

# 계정코드로 계정과목명을 찾을 수 있는 딕셔너리 생성
account_name_map = dict(
    zip(
        accounts["account_code"].astype(int),
        accounts["account_name"]
    )
)

# 변환된 ERP 업로드 행을 저장할 빈 목록
erp_rows = []


# 입력 가능 전표를 한 건씩 반복
for _, row in ready_journal.iterrows():
    line_no = 1

    # 첫 번째와 두 번째 차변 계정 처리
    debit_pairs = [
        (
            row["debit_account_1"],
            row["debit_amount_1"]
        ),
        (
            row["debit_account_2"],
            row["debit_amount_2"]
        )
    ]

    for account_value, amount_value in debit_pairs:
        # 계정코드가 없거나 금액이 0이면 ERP 행을 만들지 않음
        if (
            pd.isna(account_value)
            or pd.isna(amount_value)
            or amount_value == 0
        ):
            continue

        account_code = int(account_value)

        # 차변 계정 한 개를 ERP 입력용 한 행으로 추가
        erp_rows.append(
            {
                "voucher_id": row["voucher_id"],
                "line_no": line_no,
                "transaction_date": row["transaction_date"],
                "department_code": row["department_code"],
                "partner_code": row["partner_code"],
                "debit_credit_type": "차변",
                "account_code": account_code,
                "account_name": account_name_map.get(
                    account_code,
                    "알 수 없음"
                ),
                "amount": amount_value,
                "evidence_type": row["evidence_type"],
                "evidence_no": row["evidence_no"],
                "description": row["description"],
                "remarks": row["remarks"]
            }
        )

        line_no += 1

    # 첫 번째와 두 번째 대변 계정 처리
    credit_pairs = [
        (
            row["credit_account_1"],
            row["credit_amount_1"]
        ),
        (
            row["credit_account_2"],
            row["credit_amount_2"]
        )
    ]

    for account_value, amount_value in credit_pairs:
        # 계정코드가 없거나 금액이 0이면 ERP 행을 만들지 않음
        if (
            pd.isna(account_value)
            or pd.isna(amount_value)
            or amount_value == 0
        ):
            continue

        account_code = int(account_value)

        # 대변 계정 한 개를 ERP 입력용 한 행으로 추가
        erp_rows.append(
            {
                "voucher_id": row["voucher_id"],
                "line_no": line_no,
                "transaction_date": row["transaction_date"],
                "department_code": row["department_code"],
                "partner_code": row["partner_code"],
                "debit_credit_type": "대변",
                "account_code": account_code,
                "account_name": account_name_map.get(
                    account_code,
                    "알 수 없음"
                ),
                "amount": amount_value,
                "evidence_type": row["evidence_type"],
                "evidence_no": row["evidence_no"],
                "description": row["description"],
                "remarks": row["remarks"]
            }
        )

        line_no += 1


# ERP 행 목록을 데이터프레임으로 변환
erp_upload = pd.DataFrame(erp_rows)

print("입력 가능 전표 수:", ready_journal["voucher_id"].nunique())
print("변환된 ERP 분개 행 수:", len(erp_upload))

erp_upload

입력 가능 전표 수: 10
변환된 ERP 분개 행 수: 29


,voucher_id,line_no,transaction_date,department_code,partner_code,debit_credit_type,account_code,account_name,amount,evidence_type,evidence_no,description,remarks
0,JV202608101,1,2026-08-01,D002,V001,차변,1210,원재료,5000000,전자세금계산서,NORMAL-TAX-202608-001,고무 원재료 외상 매입,8월 원재료 구매
1,JV202608101,2,2026-08-01,D002,V001,차변,1180,부가세대급금,500000,전자세금계산서,NORMAL-TAX-202608-001,고무 원재료 외상 매입,8월 원재료 구매
2,JV202608101,3,2026-08-01,D002,V001,대변,2110,외상매입금,5500000,전자세금계산서,NORMAL-TAX-202608-001,고무 원재료 외상 매입,8월 원재료 구매
3,JV202608102,1,2026-08-02,D003,V003,차변,5130,소모품비,200000,법인카드,NORMAL-CARD-202608-001,생산 작업용 장갑 구매,생산1팀 사용
4,JV202608102,2,2026-08-02,D003,V003,차변,1180,부가세대급금,20000,법인카드,NORMAL-CARD-202608-001,생산 작업용 장갑 구매,생산1팀 사용
5,JV202608102,3,2026-08-02,D003,V003,대변,2140,카드미지급금,220000,법인카드,NORMAL-CARD-202608-001,생산 작업용 장갑 구매,생산1팀 사용
6,JV202608103,1,2026-08-03,D007,V004,차변,5130,소모품비,800000,전자세금계산서,NORMAL-TAX-202608-002,제품 포장재 외상 구매,출하용 포장재
7,JV202608103,2,2026-08-03,D007,V004,차변,1180,부가세대급금,80000,전자세금계산서,NORMAL-TAX-202608-002,제품 포장재 외상 구매,출하용 포장재
8,JV202608103,3,2026-08-03,D007,V004,대변,2110,외상매입금,880000,전자세금계산서,NORMAL-TAX-202608-002,제품 포장재 외상 구매,출하용 포장재
9,JV202608104,1,2026-08-05,D007,V005,차변,5140,운반비,600000,전자세금계산서,NORMAL-TAX-202608-003,제품 납품 운송비,8월 1주차 운송비


## 5. ERP 업로드 데이터의 차변·대변 재검증

세로형으로 변환된 ERP 업로드 데이터의 전표별 차변·대변 합계를 다시 계산한다.
변환 과정에서 발생할 수 있는 행 누락, 금액 차이, 계정과목 연결 오류를 확인한다.

In [5]:
# ERP 업로드 데이터를 전표번호와 차대구분별로 집계
erp_balance_check = (
    erp_upload
    .pivot_table(
        index="voucher_id",
        columns="debit_credit_type",
        values="amount",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# pivot_table이 생성한 열 이름 표시 제거
erp_balance_check.columns.name = None

# 전표별 차변과 대변의 차이 계산
erp_balance_check["difference"] = (
    erp_balance_check["차변"]
    - erp_balance_check["대변"]
)

# 차변과 대변의 차이가 0이면 정상으로 판정
erp_balance_check["is_balanced"] = (
    erp_balance_check["difference"] == 0
)

# 같은 전표에서 동일한 순번이 중복됐는지 검사
duplicate_line_count = erp_upload.duplicated(
    subset=[
        "voucher_id",
        "line_no"
    ]
).sum()

# 계정과목명이 기준표와 연결되지 않은 행 검사
unknown_account_count = (
    erp_upload["account_name"] == "알 수 없음"
).sum()

# 차변과 대변이 일치하지 않는 전표만 추출
unbalanced_vouchers = erp_balance_check.loc[
    erp_balance_check["is_balanced"] == False
]

print("검증 대상 전표 수:", len(erp_balance_check))
print("차변·대변 불일치 전표 수:", len(unbalanced_vouchers))
print("전표 내 중복 순번 수:", duplicate_line_count)
print("알 수 없는 계정과목 수:", unknown_account_count)

erp_balance_check

검증 대상 전표 수: 10
차변·대변 불일치 전표 수: 0
전표 내 중복 순번 수: 0
알 수 없는 계정과목 수: 0


,voucher_id,대변,차변,difference,is_balanced
0,JV202608101,5500000,5500000,0,True
1,JV202608102,220000,220000,0,True
2,JV202608103,880000,880000,0,True
3,JV202608104,660000,660000,0,True
4,JV202608105,1650000,1650000,0,True
5,JV202608106,22000000,22000000,0,True
6,JV202608107,1100000,1100000,0,True
7,JV202608108,13200000,13200000,0,True
8,JV202608109,8800000,8800000,0,True
9,JV202608110,5500000,5500000,0,True


## 6. ERP 업로드 파일과 검토 대기열 저장

검증을 통과한 분개 데이터를 ERP 업로드용 CSV와 JSON으로 저장한다.
오류 전표는 담당자 검토 대기열로 분리하고, 전체 처리 상태도 별도로 저장한다.

In [6]:
# JSON 파일을 생성하기 위한 표준 라이브러리
import json


# 최종 결과 파일의 저장 위치 설정
erp_csv_path = (
    project_root
    / "data"
    / "processed"
    / "erp_upload.csv"
)

erp_json_path = (
    project_root
    / "data"
    / "processed"
    / "erp_upload.json"
)

review_queue_path = (
    project_root
    / "data"
    / "processed"
    / "review_queue.csv"
)

transaction_status_path = (
    project_root
    / "data"
    / "processed"
    / "transaction_status.csv"
)

balance_check_path = (
    project_root
    / "data"
    / "processed"
    / "erp_balance_check.csv"
)


# ERP 업로드용 세로형 분개를 CSV로 저장
erp_upload.to_csv(
    erp_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# 담당자가 확인해야 할 오류 전표를 CSV로 저장
review_queue.to_csv(
    review_queue_path,
    index=False,
    encoding="utf-8-sig"
)

# 전체 전표의 처리 상태를 CSV로 저장
classified_journal.to_csv(
    transaction_status_path,
    index=False,
    encoding="utf-8-sig"
)

# ERP 변환 후 차변·대변 검증 결과 저장
erp_balance_check.to_csv(
    balance_check_path,
    index=False,
    encoding="utf-8-sig"
)


# 전표 단위의 JSON 데이터를 저장할 빈 목록
erp_json_records = []

# ERP 업로드 데이터를 전표번호별로 묶어 JSON 구조 생성
for voucher_id, voucher_group in erp_upload.groupby(
    "voucher_id",
    sort=True
):
    # 같은 전표에서 공통으로 사용하는 정보는 첫 번째 행에서 추출
    first_row = voucher_group.iloc[0]

    # 한 전표에 포함된 차변·대변 분개 행 생성
    journal_lines = []

    for _, line in voucher_group.iterrows():
        journal_lines.append(
            {
                "line_no": int(line["line_no"]),
                "debit_credit_type": line["debit_credit_type"],
                "account_code": int(line["account_code"]),
                "account_name": line["account_name"],
                "amount": int(line["amount"])
            }
        )

    # 전표 기본정보와 분개 행을 하나의 JSON 객체로 구성
    erp_json_records.append(
        {
            "voucher_id": voucher_id,
            "transaction_date": pd.to_datetime(
                first_row["transaction_date"]
            ).strftime("%Y-%m-%d"),
            "department_code": first_row["department_code"],
            "partner_code": first_row["partner_code"],
            "evidence": {
                "type": first_row["evidence_type"],
                "number": first_row["evidence_no"]
            },
            "description": first_row["description"],
            "remarks": first_row["remarks"],
            "journal_lines": journal_lines
        }
    )


# 한글을 그대로 유지하고 들여쓰기를 적용해 JSON 파일 저장
with open(
    erp_json_path,
    "w",
    encoding="utf-8"
) as json_file:
    json.dump(
        erp_json_records,
        json_file,
        ensure_ascii=False,
        indent=2
    )


# 최종 결과 파일 생성 여부 확인
print("ERP CSV 저장:", erp_csv_path.exists())
print("ERP JSON 저장:", erp_json_path.exists())
print("검토 대기열 저장:", review_queue_path.exists())
print("처리 상태 저장:", transaction_status_path.exists())
print("잔액 검증 결과 저장:", balance_check_path.exists())

print()
print("ERP 입력 전표 수:", erp_upload["voucher_id"].nunique())
print("ERP 분개 행 수:", len(erp_upload))
print("검토 필요 전표 수:", review_queue["voucher_id"].nunique())
print("JSON 전표 수:", len(erp_json_records))

ERP CSV 저장: True
ERP JSON 저장: True
검토 대기열 저장: True
처리 상태 저장: True
잔액 검증 결과 저장: True

ERP 입력 전표 수: 10
ERP 분개 행 수: 29
검토 필요 전표 수: 10
JSON 전표 수: 10
